**1\. Customers Who Have Purchased High-Value Items**

**Why It’s Useful:** Identifies customers who purchase expensive items, helping businesses focus on high-value customers.

In [20]:
SELECT DISTINCT c.CustomerID, c.CustomerName
FROM Sales.Customers c
INNER JOIN Sales.Orders o ON c.CustomerID = o.CustomerID
INNER JOIN Sales.OrderLines ol ON o.OrderID = ol.OrderID
INNER JOIN (
    SELECT AVG(UnitPrice) AS AvgUnitPrice
    FROM Sales.OrderLines
) AS AvgPrices ON ol.UnitPrice > AvgPrices.AvgUnitPrice;


(663 rows affected)

Total execution time: 00:00:00.031

CustomerID,CustomerName
153,"Tailspin Toys (Marcell, MN)"
970,Isidora Urias
18,"Tailspin Toys (Goffstown, NH)"
143,"Tailspin Toys (Ashtabula, OH)"
945,Hoc Tran
33,"Tailspin Toys (Boyden Arbor, SC)"
1013,Annette Hetu
935,Daevasree Samavedam
952,Chuan Wattanasin
884,Anindya Ghatak


**2\. Monthly Revenue Trends**

**Why It’s Useful:** Helps analyze sales trends over time, allowing businesses to forecast seasonal demand

In [21]:
WITH MonthlySales AS (
            SELECT YEAR(o.OrderDate) AS OrderYear, MONTH(o.OrderDate) AS OrderMonth,
                   SUM(ol.Quantity * ol.UnitPrice) AS TotalSales
            FROM Sales.Orders o
            INNER JOIN Sales.OrderLines ol ON o.OrderID = ol.OrderID
            GROUP BY YEAR(o.OrderDate), MONTH(o.OrderDate)
        )
        SELECT OrderYear, OrderMonth, TotalSales
        FROM MonthlySales
        ORDER BY OrderYear DESC, OrderMonth DESC;

(41 rows affected)

Total execution time: 00:00:00.051

OrderYear,OrderMonth,TotalSales
2016,5,5138002.65
2016,4,4739058.60
2016,3,4807110.70
2016,2,4099480.35
2016,1,4612140.45
2015,12,4607182.15
2015,11,4240612.30
2015,10,4653840.90
2015,9,4841896.70
2015,8,4070526.60


**3\. Most Profitable Customers**

**Why It’s Useful:** Helps businesses identify top-spending customers to tailor loyalty programs and other discounts/deals.

In [22]:
SELECT c.CustomerID, c.CustomerName, 
       SUM(ol.Quantity * ol.UnitPrice * (1 - ol.TaxRate / 100)) AS TotalProfit
FROM Sales.Orders o
INNER JOIN Sales.OrderLines ol ON o.OrderID = ol.OrderID
INNER JOIN Sales.Customers c ON o.CustomerID = c.CustomerID
GROUP BY c.CustomerID, c.CustomerName
HAVING SUM(ol.Quantity * ol.UnitPrice * (1 - ol.TaxRate / 100)) > 1000
ORDER BY TotalProfit DESC;


(663 rows affected)

Total execution time: 00:00:00.098

CustomerID,CustomerName,TotalProfit
149,"Tailspin Toys (Inguadona, MN)",326867.727500
132,"Tailspin Toys (Minidoka, ID)",322861.715000
977,Mauno Laurila,320703.670000
580,"Wingtip Toys (Sarversville, PA)",316737.500000
964,Ingrida Zeltina,312877.852500
14,"Tailspin Toys (Long Meadow, MD)",312272.325000
954,Nasrin Omidzadeh,312076.907500
593,"Wingtip Toys (Cuyamungue, NM)",311207.472500
472,"Wingtip Toys (San Jacinto, CA)",310593.767500
550,"Wingtip Toys (Morrison Bluff, AR)",306739.560000


**4\. Late Deliveries (Orders Delivered After Expected Date)**

**Why It’s Useful:** Allows businesses to identify delivery inefficiencies and improve logistics. For example, recognizing that orders that are delivered a day or more late are with a certain shipping company.

In [23]:
SELECT o.OrderID, c.CustomerName, o.OrderDate, o.ExpectedDeliveryDate, d.ConfirmedDeliveryTime
FROM Sales.Orders o
INNER JOIN Sales.Customers c ON o.CustomerID = c.CustomerID
LEFT JOIN Sales.Invoices d ON o.OrderID = d.OrderID
WHERE DATEDIFF(DAY, o.ExpectedDeliveryDate, d.ConfirmedDeliveryTime) >= 1;


(4470 rows affected)

Total execution time: 00:00:00.314

OrderID,CustomerName,OrderDate,ExpectedDeliveryDate,ConfirmedDeliveryTime
18,"Wingtip Toys (Baldwin City, KS)",2013-01-01,2013-01-02,2013-01-03 07:05:00.0000000
21,"Wingtip Toys (Cowlington, OK)",2013-01-01,2013-01-02,2013-01-03 07:10:00.0000000
31,"Wingtip Toys (Mahaffey, PA)",2013-01-01,2013-01-02,2013-01-03 07:15:00.0000000
45,Aakriti Byrraju,2013-01-01,2013-01-02,2013-01-03 07:20:00.0000000
46,Bala Dixit,2013-01-01,2013-01-02,2013-01-03 07:25:00.0000000
47,"Tailspin Toys (Tomnolen, MS)",2013-01-01,2013-01-02,2013-01-03 07:30:00.0000000
48,Sara Huiting,2013-01-01,2013-01-02,2013-01-03 07:35:00.0000000
49,"Wingtip Toys (Trumansburg, NY)",2013-01-01,2013-01-02,2013-01-03 07:40:00.0000000
50,Ingrida Zeltina,2013-01-01,2013-01-02,2013-01-03 07:45:00.0000000
51,"Tailspin Toys (Hahira, GA)",2013-01-01,2013-01-02,2013-01-03 07:50:00.0000000


**5\. Employees with the Most Orders Processed**

**Why It’s Useful:** Helps recognize top-performing employees in order processing. Allows for analysis of salesperson performance.

In [24]:
SELECT e.PersonID, e.FullName, COUNT(o.OrderID) AS TotalOrdersProcessed
FROM Sales.Orders o
INNER JOIN Application.People e ON o.SalespersonPersonID = e.PersonID
GROUP BY e.PersonID, e.FullName
ORDER BY TotalOrdersProcessed DESC;


(10 rows affected)

Total execution time: 00:00:00.007

PersonID,FullName,TotalOrdersProcessed
16,Archer Lamble,7532
2,Kayla Woodcock,7474
13,Hudson Hollinworth,7400
20,Jack Potter,7387
15,Taj Shand,7371
6,Sophia Hinton,7349
3,Hudson Onslow,7281
7,Amy Trefl,7276
14,Lily Code,7268
8,Anthony Grosse,7257


**6\. Most Frequently Ordered Product per Customer**

**Why It’s Useful:** Helps businesses understand customer preferences for targeted marketing. For example, if someone orders multiple of a mug, then the company can advertise other mugs to them.

In [25]:
WITH CustomerProductOrders AS (
    SELECT o.CustomerID, c.CustomerName, ol.StockItemID, si.StockItemName, COUNT(ol.StockItemID) AS OrderCount
    FROM Sales.Orders o
    INNER JOIN Sales.OrderLines ol ON o.OrderID = ol.OrderID
    INNER JOIN Sales.Customers c ON o.CustomerID = c.CustomerID
    INNER JOIN Warehouse.StockItems si ON ol.StockItemID = si.StockItemID
    GROUP BY o.CustomerID, c.CustomerName, ol.StockItemID, si.StockItemName
)
SELECT CustomerID, CustomerName, StockItemID, StockItemName, OrderCount
FROM CustomerProductOrders
WHERE OrderCount = (SELECT MAX(OrderCount) FROM CustomerProductOrders CPO WHERE CPO.CustomerID = CustomerProductOrders.CustomerID);


(1475 rows affected)

Total execution time: 00:00:00.093

CustomerID,CustomerName,StockItemID,StockItemName,OrderCount
1036,Erik Malk,46,Developer joke mug - a foo walks into a bar (White),4
401,Wingtip Toys (Head Office),131,Furry gorilla with big eyes slippers (Black) M,9
953,Linh Dao,125,Ogre battery-powered slippers (Green) XL,6
418,"Wingtip Toys (Yaak, MT)",131,Furry gorilla with big eyes slippers (Black) M,6
199,"Tailspin Toys (Antonito, CO)",68,Ride on toy sedan car (Red) 1/12 scale,5
921,Victoria Lacusta,151,Pack of 12 action figures (male),4
965,Phoung Cu,156,10 mm Double sided bubble wrap 10m,5
120,"Tailspin Toys (Bratenahl, OH)",162,32 mm Double sided bubble wrap 10m,5
114,"Tailspin Toys (Cherry Grove Beach, SC)",98,"""The Gu"" red shirt XML tag t-shirt (Black) 4XL",5
900,Lilli Sokk,76,"""The Gu"" red shirt XML tag t-shirt (White) 3XS",8


**7\. Top 10 Most Popular Products**

**Why It’s Useful:** Identifies the products with the highest quantity sold, seeing which products have performed the best long-term.

In [26]:
SELECT TOP 10 si.StockItemID, si.StockItemName, SUM(ol.Quantity) AS TotalQuantitySold
FROM Sales.OrderLines ol
INNER JOIN Warehouse.StockItems si ON ol.StockItemID = si.StockItemID
GROUP BY si.StockItemID, si.StockItemName
ORDER BY TotalQuantitySold DESC;


(10 rows affected)

Total execution time: 00:00:00.005

StockItemID,StockItemName,TotalQuantitySold
191,Black and orange fragile despatch tape 48mmx75m,207324
192,Black and orange fragile despatch tape 48mmx100m,193680
189,Clear packaging tape 48mmx75m,158626
188,3 kg Courier post bag (White) 300x190x95mm,152375
185,Shipping carton (Brown) 356x356x279mm,152125
184,Shipping carton (Brown) 305x305x305mm,151875
187,Express post box 5kg (White) 350x280x130mm,149825
177,Shipping carton (Brown) 413x285x187mm,147675
179,Shipping carton (Brown) 229x229x229mm,146375
186,Shipping carton (Brown) 457x457x457mm,144950


**8\. Customers Who Haven’t Ordered in Over a Year**

**Why It’s Useful:** Helps businesses recognize which customers haven't ordered in a while, allowing them to re-engage these customers with advertisements.

In [27]:
SELECT c.CustomerID, c.CustomerName, LastOrder.LastOrderDate
        FROM Sales.Customers c
        OUTER APPLY (
            SELECT TOP 1 o.OrderDate AS LastOrderDate
            FROM Sales.Orders o
            WHERE o.CustomerID = c.CustomerID
            ORDER BY o.OrderDate DESC
        ) AS LastOrder
        WHERE DATEDIFF(DAY, LastOrder.LastOrderDate, GETDATE()) > 365;

(663 rows affected)

Total execution time: 00:00:00.096

CustomerID,CustomerName,LastOrderDate
1,Tailspin Toys (Head Office),2016-05-27
2,"Tailspin Toys (Sylvanite, MT)",2016-05-14
3,"Tailspin Toys (Peeples Valley, AZ)",2016-05-30
4,"Tailspin Toys (Medicine Lodge, KS)",2016-04-28
5,"Tailspin Toys (Gasport, NY)",2016-05-28
6,"Tailspin Toys (Jessie, ND)",2016-05-31
7,"Tailspin Toys (Frankewing, TN)",2016-05-27
8,"Tailspin Toys (Bow Mar, CO)",2016-05-20
9,"Tailspin Toys (Netcong, NJ)",2016-05-24
10,"Tailspin Toys (Wimbledon, ND)",2016-05-24


**9\. Most Recent Order for Each Customer**

**Why It’s Useful:** This helps businesses quickly identify the latest interactions with customers, useful for follow-ups and trend analysis.

In [28]:
SELECT c.CustomerID, c.CustomerName, o.OrderID, o.OrderDate
FROM Sales.Customers c
OUTER APPLY (
    SELECT TOP 1 o.OrderID, o.OrderDate
    FROM Sales.Orders o
    WHERE o.CustomerID = c.CustomerID
    ORDER BY o.OrderDate DESC
) AS o
ORDER BY o.OrderDate DESC;


(663 rows affected)

Total execution time: 00:00:00.100

CustomerID,CustomerName,OrderID,OrderDate
6,"Tailspin Toys (Jessie, ND)",73547,2016-05-31
11,"Tailspin Toys (Devault, PA)",73573,2016-05-31
28,"Tailspin Toys (North Ridge, NY)",73519,2016-05-31
29,"Tailspin Toys (Eulaton, AL)",73557,2016-05-31
35,"Tailspin Toys (Slanesville, WV)",73507,2016-05-31
64,"Tailspin Toys (Hodgdon, ME)",73561,2016-05-31
76,"Tailspin Toys (Yewed, OK)",73513,2016-05-31
82,"Tailspin Toys (La Cueva, NM)",73538,2016-05-31
87,"Tailspin Toys (Sauquoit, NY)",73522,2016-05-31
90,"Tailspin Toys (Tolna, ND)",73565,2016-05-31


**10\. Top 3 Most Recent Orders Per Employee**

**Why It’s Useful:** Identifies employees handling the most recent transactions, useful for performance tracking and workload distribution.

In [29]:
SELECT e.PersonID AS EmployeeID, e.FullName, o.OrderID, o.OrderDate
FROM Application.People e
CROSS APPLY (
    SELECT TOP 3 o.OrderID, o.OrderDate
    FROM Sales.Orders o
    WHERE o.PickedByPersonID = e.PersonID
    ORDER BY o.OrderDate DESC
) AS o
ORDER BY e.FullName, o.OrderDate DESC;


(57 rows affected)

Total execution time: 00:00:00.105

EmployeeID,FullName,OrderID,OrderDate
9,Alica Fatnowna,68339,2016-03-12
9,Alica Fatnowna,68340,2016-03-12
9,Alica Fatnowna,68341,2016-03-12
7,Amy Trefl,73107,2016-05-25
7,Amy Trefl,73108,2016-05-25
7,Amy Trefl,73109,2016-05-25
8,Anthony Grosse,73172,2016-05-26
8,Anthony Grosse,73173,2016-05-26
8,Anthony Grosse,73174,2016-05-26
16,Archer Lamble,72278,2016-05-12
